# JCM v1.0 — Running the Base Model with ERA5 Initial Conditions

This notebook demonstrates how to:
1. Load ERA5 reanalysis data (via WeatherBench2)
2. Interpolate ERA5 fields onto the JCM grid
3. Convert the interpolated ERA5 fields into a `PhysicsState` dataclass
4. Run a short simulation

**Prerequisites:** `jcm`, `jax`, `xarray`, `numpy`, `pandas`,`gcsfs`

In [ ]:
import jax.numpy as jnp
import xarray as xr
import pandas as pd

from jcm.model import Model
from jcm.utils import get_speedy_coords
from jcm.physics_interface import PhysicsState

## 1. Load ERA5

Open the Zarr store and rename dimensions/variables to match the JCM naming
conventions (`lat`, `lon`, `u_wind`, `v_wind`).

In [ ]:
era5_ds = xr.open_zarr(
    "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-64x32_equiangular_conservative.zarr",
    consolidated=True,
    storage_options={"token": "anon"},
)

era5_ds = era5_ds.rename({
    "latitude":             "lat",
    "longitude":            "lon",
    "u_component_of_wind":  "u_wind",
    "v_component_of_wind":  "v_wind",
})

era5_ds

## 2. Interpolate ERA5 → JCM grid

The function below takes an arbitrary time slice of ERA5, interpolates 3-D
prognostic fields (`temperature`, `specific_humidity`, `u_wind`, `v_wind`,
`geopotential`) onto the model's pressure levels *and* horizontal grid, and
separately interpolates 2-D fields (`surface_pressure`) onto the horizontal
grid only.

In [ ]:
# Get the default SPEEDY grid
coords = get_speedy_coords()

MODEL_LEVELS = coords.vertical.centers
MODEL_LATS   = coords.horizontal.latitudes
MODEL_LONS   = coords.horizontal.longitudes

VARS_3D = ["temperature", "specific_humidity", "u_wind", "v_wind", "geopotential"]
VARS_2D = ["surface_pressure"]


def interpolate_era5(time_slice):
    """Interpolate ERA5 fields onto the JCM grid.

    Parameters
    ----------
    time_slice : slice or scalar
        Passed to ``era5_ds.sel(time=...)``.

    Returns
    
    xr.Dataset
        Merged dataset on the JCM grid.
    """
    ds = era5_ds.sel(time=time_slice)

    ds_3d = ds[VARS_3D].interp(
        {"level": MODEL_LEVELS, "lat": MODEL_LATS, "lon": MODEL_LONS},
        kwargs={"fill_value": "extrapolate"},
    )

    ds_2d = ds[VARS_2D].interp(
        {"lat": MODEL_LATS, "lon": MODEL_LONS},
        kwargs={"fill_value": "extrapolate"},
    )

    return xr.merge([ds_3d, ds_2d])

## 3. Convert interpolated ERA5 to a `PhysicsState`

Pack the interpolated xarray fields into the JAX-backed `PhysicsState`
dataclass that JCM expects as initial conditions.  Surface pressure is
derived from the model's default modal state and then standardised.

In [ ]:
def era5_to_physics_state(ds_interpolated: xr.Dataset) -> PhysicsState:
    """Build a PhysicsState from a single-time interpolated ERA5 dataset."""
    T = jnp.asarray(ds_interpolated["temperature"].values)
    q = jnp.asarray(ds_interpolated["specific_humidity"].values)
    u = jnp.asarray(ds_interpolated["u_wind"].values)
    v = jnp.asarray(ds_interpolated["v_wind"].values)
    g = jnp.asarray(ds_interpolated["geopotential"].values)
    sp = jnp.asarray(ds_interpolated["surface_pressure"].values)
    nsp = sp / 1e5  # Normalize surface pressure by a reference pressure ## CHECK ME

    return PhysicsState(
        u_wind=u,
        v_wind=v,
        temperature=T,
        specific_humidity=q,
        geopotential=g,
        normalized_surface_pressure=nsp,
    )

## 4. ERA5-initialised simulation

Combines interpolation, state conversion, and the model forward pass into a
single call.

In [ ]:
init_time = pd.Timestamp("2000-01-01T00:00:00")
era5_init  = interpolate_era5(time_slice=init_time)
init_state = era5_to_physics_state(era5_init)

model = Model(coords=coords)

predictions = model.run(
    initial_state=init_state,
    total_time=30,
    save_interval=1.0,  
)

pred_ds = predictions.to_xarray()
print(pred_ds)